# 01 - Exploratory data analysis: PaySim mobile-money transactions

PaySim is a synthetic dataset built by simulating one month of mobile-money activity. Goal here: understand where fraud occurs, how rare it is, and which patterns an AML analyst could turn into rules.

Run from the `notebooks/` folder after placing the CSV in `data/paysim.csv`. Set `NROWS` for a quicker run on a laptop.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_PATH, LARGE_TRANSFER, TARGET
from src.data import load_data

NROWS = None   # for example 1_000_000 for a faster, smaller run
df = load_data(DATA_PATH, nrows=NROWS)
df.head()

## 1. Size and class balance

In [ ]:
print(df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print(df[TARGET].value_counts().rename({0: "legit", 1: "fraud"}))
print(f"Fraud rate: {df[TARGET].mean():.4%}")
print("Steps:", df["step"].min(), "to", df["step"].max(), "(1 step = 1 hour)")

Fraud is extremely rare, so accuracy is meaningless and the evaluation must use precision-recall metrics and recall at a fixed false-alert rate.

## 2. Where does fraud happen?

In [ ]:
by_type = df.groupby("type", observed=True)[TARGET].agg(transactions="count", fraud="sum")
by_type["fraud_rate_%"] = by_type["fraud"] / by_type["transactions"] * 100
by_type

In the real PaySim data fraud appears only in `TRANSFER` and `CASH_OUT`, mirroring the fraud pattern the simulator injects (take money out via a transfer, then cash it out). Any model or rule should focus on those two types.

## 3. The built-in rule: `isFlaggedFraud`

The simulator flags single transfers above 200,000. How much real fraud does that rule catch?

In [ ]:
flag = df["isFlaggedFraud"] == 1
print("Flagged by the built-in rule:", int(flag.sum()))
print("Of which actually fraud    :", int((flag & (df[TARGET] == 1)).sum()))
print(f"Rule recall: {(flag & (df[TARGET] == 1)).sum() / df[TARGET].sum():.2%}")
big = (df["type"].astype(str) == "TRANSFER") & (df["amount"] > LARGE_TRANSFER)
print("Transfers over 200,000     :", int(big.sum()), "| fraud among them:", int((big & (df[TARGET] == 1)).sum()))

A single large-transfer rule misses almost all fraud. That is the gap a model (and richer rules) needs to close.

## 4. Amounts

In [ ]:
risky = df[df["type"].astype(str).isin(["TRANSFER", "CASH_OUT"])]
print(risky.groupby(TARGET)["amount"].describe().round(0))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, col, lab in [(0, "#4C78A8", "Legit"), (1, "#E45756", "Fraud")]:
    ax[0].hist(np.log1p(risky.loc[risky[TARGET] == k, "amount"]), bins=50, alpha=0.6, color=col, label=lab, density=True)
ax[0].set_title("log(1 + amount), TRANSFER and CASH_OUT"); ax[0].legend()
risky.boxplot(column="amount", by=TARGET, ax=ax[1], showfliers=False)
ax[1].set_title("Amount by class (no outliers)"); plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Time patterns

In [ ]:
hour = (df["step"].astype("int32") - 1) % 24
day = (df["step"].astype("int32") - 1) // 24
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
(df.groupby(hour)[TARGET].mean() * 100).plot.bar(ax=ax[0], color="#E45756"); ax[0].set_title("Fraud rate (%) by hour of day")
df.groupby(day)[TARGET].sum().plot.bar(ax=ax[1], color="#F58518"); ax[1].set_title("Fraud cases per day")
plt.tight_layout()
plt.show()

## 6. Account-drain pattern

The simulator's fraudsters try to empty accounts. Compare the amount with the sender's balance **before** the transaction (known at decision time).

In [ ]:
r = risky.copy()
r["drains_account"] = (r["oldbalanceOrg"] > 0) & (r["amount"] >= r["oldbalanceOrg"])
tab = r.groupby("drains_account")[TARGET].agg(transactions="count", fraud="sum")
tab["fraud_rate_%"] = tab["fraud"] / tab["transactions"] * 100
tab

## 7. Destination accounts

Merchant accounts start with `M`. Check how often destination accounts are reused (mule-style behaviour would show up as repeated receivers).

In [ ]:
dest = df["nameDest"].astype(str)
print("Distinct destination accounts:", dest.nunique())
print("Merchant share of transactions:", round(float(dest.str.startswith("M").mean()), 3))
counts = dest.value_counts()
print("Destinations used more than once:", int((counts > 1).sum()))
fraud_dest = df.loc[df[TARGET] == 1, "nameDest"].astype(str).value_counts()
print("Fraud destinations used more than once:", int((fraud_dest > 1).sum()), "of", len(fraud_dest))

## Takeaways

- Fraud is a tiny share of transactions and sits entirely in `TRANSFER` and `CASH_OUT` (check the table in section 2 on your data).
- The simulator's own large-transfer rule catches very little fraud.
- Draining the sender's account is a strong signal that is known before the transaction completes.
- Sender accounts rarely repeat in PaySim, so sender-side velocity features are not informative here; destination history is the only account history worth building.
- The balance-after columns can leak the label in this simulation (see `02_experiments.ipynb`), so the main model avoids them.